# MinIOの接続パラメータを Connectionから取得

In [36]:
import os

endpoint_url = "http://" + os.environ['AWS_S3_ENDPOINT']
# s3bucket = os.environ['s3bucket_models']
bucket_models = os.environ['AWS_S3_BUCKET']
access_key = os.environ['AWS_ACCESS_KEY_ID']
secret_key = os.environ['AWS_SECRET_ACCESS_KEY']

bucket_models = "models"
base_prefix = "demo/"

# ローカルファイルとMinIO上のファイル名
local_files = {
    "clip-vit-base-patch16_int8_lowacc.xml": "demo.xml",
    "clip-vit-base-patch16_int8_lowacc.bin": "demo.bin"
}

# MinIOに接続

In [37]:
import os
import boto3
from botocore.client import Config

# boto3クライアント
s3 = boto3.client(
    's3',
    endpoint_url=endpoint_url,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name='us-east-1',
    config=Config(signature_version='s3v4')
)


# demo/配下のディレクトリ有無の調査

In [38]:
# demo/ 以下の既存ディレクトリ調査
response = s3.list_objects_v2(Bucket=bucket_models, Prefix=base_prefix)
version = 0

if "Contents" in response:
    for obj in response['Contents']:
        parts = obj['Key'].split('/')
        if len(parts) >= 2 and parts[1].isdigit():
            ver = int(parts[1])
            if ver > version:
                version = ver
else:
    print("No existing versioned directory found under demo/")

# 指定したモデルをMinIOにアップロード

In [39]:
# 次のバージョン番号を決定
version += 1
new_prefix = f"{base_prefix}{version}/"
print(f"Uploading to: {new_prefix}")

# 空オブジェクトで"ディレクトリ"を作成
s3.put_object(Bucket=bucket_models, Key=new_prefix)

# ファイルをアップロード（名前をdemo.xml / demo.binに変更）
for local_path, remote_name in local_files.items():
    if os.path.exists(local_path):
        with open(local_path, 'rb') as f:
            s3.upload_fileobj(f, bucket_models, f"{new_prefix}{remote_name}")
        print(f"Uploaded: {local_path} → {new_prefix}{remote_name}")
    else:
        print(f"File not found: {local_path}")

Uploading to: demo/2/
Uploaded: clip-vit-base-patch16_int8.xml → demo/2/demo.xml
Uploaded: clip-vit-base-patch16_int8.bin → demo/2/demo.bin
